<h1><center>Laboratorio 12: Benchmark de Carga y Modelos con Spotify 🎵</center></h1>

<center><strong>IA7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

### Cuerpo Docente

- Profesores: Pablo Badilla e Ignacio Núñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Carrasco, Ignacio Reyes

### Equipo

**Los notebooks sin los nombres del equipo no se revisan.**

- Nombre de alumno 1:
- Nombre de alumno 2:

### Reglas

- Grupos de dos personas.
- Las dudas fuera del horario de clases se publican en el foro; el equipo docente responde por ese medio.
- Pueden usar LLM y asistentes de código ([Copilot](https://docs.github.com/en/copilot), [Claude](https://claude.ai/), [Cursor](https://cursor.com/docs), etc.) solo para hacer consultas, leer documentación y corregir errores; no para escribir el código ni las respuestas que se evalúan.

### Flujo de trabajo

Este laboratorio es experimental: se ejecuta, se mide y se interpreta lo medido. Cada sección sigue el mismo ciclo. Primero ejecutan el código dado, que viene identificado como tal. Luego completan lo que pide el recuadro **Ejercicio**, cuando lo hay: las secciones 1.1, 1.2, 1.3 y 3 dejan código por completar; las secciones 2.1 y 2.2 traen el código resuelto, y 2.3 permite medir una GPU compatible o usar la referencia docente. Por último responden las preguntas con lo que observaron en sus propias mediciones.

Los tiempos medidos dependen de su máquina: citen los números que obtuvieron. En 2.3 distingan sus mediciones, la referencia docente y los tiempos hipotéticos de las preguntas; no presenten datos provistos como mediciones propias. Al terminar cada sección, guarden el notebook y hagan commit en su rama. La entrega es una sola pull request con el notebook, en la fecha publicada.

### Configuración del ambiente

Desde la carpeta del laboratorio, instalen con `uv` las librerías declaradas en `pyproject.toml`:

```bash
uv sync
```

Ese archivo ya incluye `pandas`, `polars`, `numba`, `scikit-learn` y `plotly`; la instalación opcional de JAX para GPU se explica en 2.3.

## Contexto

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Radio Faro Papaya</strong>

  <img src="assets/lore.png" alt="Pía, la locutora de Radio Faro Papaya, habla por micrófono con audífonos en una mesa sobre la playa de La Serena, con el Faro Monumental al fondo, un jugo con hielo, papayas en una canasta y globos aerostáticos con forma de botellas de pisco, aceite de oliva y néctar de papaya." style="display:block; margin:8px auto 16px; width:800px; max-width:100%; height:auto;">

Hernán Rojas dirige Radio Faro Papaya, una radio comunitaria que transmite desde una mesa en la playa, a los pies del Faro Monumental, en La Serena. La atienden tres voluntarios y un solo computador. Pía, la locutora de las mañanas, lo usa para armar su programa. Toño, el voluntario que «sabe de computadores» desde que una vez arregló una impresora, lo usa para la transmisión en vivo. Hernán lo usa para ordenar el catálogo, que es el orgullo de la radio: 200 000 canciones juntadas durante años a punta de donaciones de oyentes.

Hernán quiere dejar de programar a ojo. En verano La Serena se llena de gente y de planes (playa de día, paseo por la costanera al atardecer, cielo despejado por la noche) y le gustaría que la música acompañara cada franja: canciones más alegres cuando el sol pega fuerte y más tranquilas cuando cae la noche. Para eso necesita saber qué tan alegre suena cada canción del catálogo, y escucharlas una por una no es opción.

Cuando el computador abre los 20 archivos del catálogo se traba, y una vez, en pleno programa de Pía, la radio quedó en silencio al aire. Hernán no quiere repetirlo en temporada alta, así que le pidió ayuda a ustedes, el equipo de datos que Pía consiguió. Necesita un sistema estable: que cargue el catálogo sin trabar el computador, que calcule rápido qué tan alegre es una canción nueva y que, si hoy dice que un tema es alegre, mañana diga lo mismo. Como en la radio nadie tiene presupuesto para equivocarse, quiere ver las mediciones antes de apoyarse en el resultado.
</div>

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`, en `data/raw/`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición: las columnas del dataset</strong>

Cada canción tiene 24 columnas. Estas son las que usaremos; las medidas de audio las calcula Spotify. Los rangos corresponden a los valores que aparecen en los 20 archivos.

| Columna | Qué mide | Rango en los datos |
| --- | --- | --- |
| `valence` | Positividad musical: cuanto más alta, más alegre suena la canción. Es lo que vamos a predecir. | 0 a 1 |
| `danceability` | Qué tan adecuada es la canción para bailar. | 0,05 a 0,99 |
| `energy` | Intensidad y actividad que se perciben al escucharla. | 0 a 1 |
| `loudness` | Volumen promedio, en decibeles (dB). 0 dB es el máximo, por eso casi todos los valores son negativos. | −42,5 a 0 |
| `speechiness` | Presencia de palabras habladas. Valores altos indican más habla, como en un podcast. | 0,02 a 0,96 |
| `acousticness` | Confianza de que la canción sea acústica. | 0 a 1 |
| `instrumentalness` | Confianza de que la canción no tenga voces. | 0 a 1 |
| `liveness` | Confianza de que se grabó con público, es decir, en vivo. | 0,01 a 1 |
| `tempo` | Pulsos por minuto (BPM). | 32,4 a 245,9 |
| `duration_ms` | Duración, en milisegundos. | 15 147 a 1 047 307 |
| `key` | Tonalidad, codificada de 0 (do) a 11 (si). | 0 a 11 |
| `mode` | Modo de la escala: 1 es mayor y 0, menor. | 0 y 1 |
| `genre` | Género de la canción: Blues, Classical, Country, Electronic, Folk, Hip-Hop, Jazz, Pop, R&B o Rock. | 10 categorías |
| `year` | Año de lanzamiento de la canción. | 1900 a 2025 |
| `popularity` | Índice de popularidad de la canción. | 0 a 96 |
| `total_artist_followers` | Seguidores totales de los artistas de la canción. | 0 a 295 181 876 |
| `avg_artist_popularity` | Popularidad promedio de los artistas de la canción. | 0 a 100 |

`key` y `mode` son números enteros, pero representan categorías: promediar tonalidades no tiene sentido.
</div>

## Objetivos

- Cargar los datos de canciones desde archivos Parquet y reducir su uso de memoria ajustando los tipos de datos y leyendo solo las columnas necesarias.
- Comparar lectura en serie, con hilos y con procesos, distinguiendo sus costos y modelos de memoria.
- Consultar un servicio local simulado con async y comprobar la equivalencia entre consultas secuenciales y concurrentes.
- Analizar cómo cambia el tiempo de predicción de un modelo lineal según su implementación: Python puro, NumPy, Numba, pandas y Polars.
- Entrenar un `RandomForestRegressor` que prediga la valencia y comparar el efecto de paralelizar el entrenamiento con `n_jobs`.
- Evaluar una propuesta GPU considerando transferencias, memoria y precisión, usando mediciones propias o una referencia docente.
- Combinar resultados de particiones desiguales y explicar qué cambia al repartir el trabajo entre máquinas.

## Temas a tratar

- Lectura de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos y lectura de solo las columnas necesarias.
- Hilos, procesos y concurrencia cooperativa con `asyncio`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con `RandomForestRegressor`, efecto de `n_jobs` y comparación con modelos de referencia.
- Costo total y medición de operaciones GPU.
- Particiones, reducción y comunicación en computación distribuida.

<!-- EVALUACIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #8b5cf6; background:rgba(139,92,246,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📊 Evaluación</strong>
  El laboratorio vale <strong>6,0 puntos</strong>. La nota se calcula como <code>1,0 + puntaje obtenido</code>: 0,0 puntos corresponden a nota 1,0 y 6,0 puntos a nota 7,0. Cada sección indica su puntaje.
</div>

| Sección | Contenido | Puntaje |
|---|---|---:|
| 1.1 | Optimización de tipos y lectura por columnas | 1,0 |
| 1.2 | Lectura con hilos y procesos | 1,0 |
| 1.3 | Async local | 0,5 |
| 2.1 | Regresión lineal a mano | 1,5 |
| 2.2 | Entrenamiento y `n_jobs` | 1,0 |
| 2.3 | Decisiones sobre GPU | 0,5 |
| 3.1 | Computación distribuida: simulación local | 0,5 |
| **Total** | | **6,0** |

### Rutas del laboratorio

**Código dado:** esta celda calcula la raíz del laboratorio y la carpeta de los datos. El notebook se ejecuta desde `notebooks/`, por eso la raíz es su carpeta madre.

In [ ]:
import sys
from pathlib import Path

RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

RAW_DIR = RUTA_LAB / "data" / "raw"

## 1. Carga y Optimización de Datos con Parquet

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Bits, bytes y unidades de medida</strong>

Un **bit** puede valer 0 o 1; un **byte** agrupa 8 bits. Usaremos `b` para bits y `B` para bytes: un valor de 32 bits ocupa 4 bytes. Para expresar muchos bytes hay dos convenciones:

| Unidad decimal | Bytes | Unidad binaria | Bytes |
|---|---:|---|---:|
| kB | 1000 | KiB | 1024 |
| MB | 1 000 000 | MiB | 1 048 576 (`2**20`) |
| GB | 1 000 000 000 | GiB | 1 073 741 824 (`2**30`) |

Una cantidad de bytes describe espacio; **MB/s** describe cuántos megabytes se transfieren por segundo. **Mb/s** expresa megabits por segundo: ocho bits forman un byte, por lo que 80 Mb/s equivalen a 10 MB/s antes de otros costos de transmisión.

Para tiempos usaremos segundos (`s`), milisegundos (`ms`, una milésima de segundo), microsegundos (`µs`, una millonésima) y nanosegundos (`ns`, una milmillonésima). Así, 1 ms = 1000 µs = 1 000 000 ns. Comparen tiempos después de convertirlos a la misma unidad.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 CPU: ejecutar las instrucciones del programa</strong>

La **CPU** (*Central Processing Unit*, unidad central de procesamiento) ejecuta instrucciones: realiza operaciones aritméticas, compara valores y controla qué instrucciones siguen. Cuando Python lee un archivo o transforma una tabla, la CPU ejecuta el código del intérprete y de las bibliotecas que este llama. El disco puede estar entregando datos mientras la CPU espera o trabaja en otra tarea; el tiempo de una operación incluye esas esperas, además del cálculo.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 RAM y almacenamiento persistente</strong>

La **RAM** es la memoria de trabajo: allí mantenemos los datos que los programas están usando. La RAM habitual del computador pierde su contenido al apagarlo. Un **SSD** o disco guarda archivos de forma persistente, incluso sin energía; sus datos deben llegar a memoria para que el programa los procese.

Acceder a RAM suele tomar mucho menos tiempo que solicitar datos al almacenamiento. Como orientación sobre la espera de un acceso, no sobre la lectura de un archivo completo:

| Recurso | Orden de magnitud orientativo |
|---|---|
| RAM | Decenas a centenas de ns |
| Almacenamiento flash de un SSD | Decenas a centenas de µs |

Por ejemplo, 100 µs son 1000 veces 100 ns. Los valores reales dependen del equipo y del acceso; [Western Digital compara estas escalas para DRAM y NAND](https://www.westerndigital.com/en-sg/company/innovation/non-volatile-memory). Esta diferencia no implica que un programa completo se acelere 1000 veces al tener sus datos en RAM: también debe transformar datos, reservar memoria y construir resultados.

En nuestro caso, leer Parquet incluye obtener los bytes y decodificarlos. Además, el sistema operativo puede conservar archivos recién leídos en RAM: una segunda lectura puede usar esa caché de archivos y tardar menos sin que el SSD haya cambiado. Esa caché es distinta de la memoria pequeña y rápida que la propia CPU usa para datos e instrucciones.
<img src="assets/ram-almacenamiento-cache.svg" alt="El archivo persiste en SSD; la RAM contiene una caché de sus bytes y la tabla decodificada. La CPU ejecuta la decodificación y el procesamiento y tiene una caché distinta." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

### 1.1 Exploración y Optimización de Tipos de Datos (1,0 puntos)

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición: cómo se codifican números y texto</strong>

Los mismos bits pueden interpretarse como números o texto según cómo se codificaron. Veamos qué cambia entre esas representaciones.

**Enteros.** Con *n* bits hay 2ⁿ patrones distintos. En un entero con signo, el primer bit pesa −2ⁿ⁻¹ en vez de +2ⁿ⁻¹ (esta regla se llama *complemento a dos*). Con 16 bits hay 65 536 patrones, repartidos entre −32 768 (`1000 0000 0000 0000`) y 32 767 (`0111 1111 1111 1111`). Al sumar 1 a 32 767 cambian los 16 bits y el resultado se lee como −32 768: el desborde no produce un error, solo otro valor.

**Flotantes.** Un número con parte fraccionaria se guarda como en la notación científica: un signo, un exponente y una mantisa, que son los dígitos significativos. `float32` reparte sus 32 bits en 1 de signo, 8 de exponente y 23 de mantisa; `float64` reparte 64 en 1, 11 y 52. Como la mantisa tiene un largo fijo, la mayoría de los decimales se guarda solo de forma aproximada: en `float64`, `0.1 + 0.2` da `0.30000000000000004`, porque ni `0.1` ni `0.2` caben exactos. Más bits de mantisa significan más dígitos correctos.

<img src="assets/codificacion-numeros.svg" alt="Arriba, los 16 bits de 32 767 y de −32 768 en un int16: sumar 1 al mayor da el menor. Abajo, cómo float32 (1, 8 y 23 bits) y float64 (1, 11 y 52 bits) reparten sus bits entre signo, exponente y mantisa." style="display:block; margin:12px auto; width:720px; max-width:100%; height:auto;">

**Caracteres y strings.** Un carácter también es un número: su *punto de código* en el estándar **Unicode** (`a` es U+0061, `ñ` es U+00F1 y `♪` es U+266A). Una **codificación** define cómo escribir ese número en bytes. La más usada en archivos y redes es **UTF-8**, que ocupa 1 byte para las letras sin tilde, los dígitos y los signos básicos (el antiguo ASCII) y entre 2 y 4 bytes para el resto: `"a".encode()` mide 1 byte, `"ñ".encode()`, 2, `"♪".encode()`, 3 y `"🎵".encode()`, 4. Un **string** es una secuencia de caracteres, así que su tamaño depende de cuáles contenga; en memoria, Python usa para ello una representación distinta de UTF-8.

<img src="assets/codificacion-texto.svg" alt="Los caracteres a, ñ y ♪ con su punto de código Unicode y sus bytes en UTF-8: 1, 2 y 3 bytes." style="display:block; margin:12px auto; width:720px; max-width:100%; height:auto;">

**Cuándo se justifica cada tipo de número**

| Tipo | Se justifica cuando… | Ejemplos |
| --- | --- | --- |
| `uint8` (0 a 255) | el valor no es negativo y cabe en un byte | el canal de color de un píxel |
| `int8` (−128 a 127) | los valores son pocos y muy acotados | una nota en décimas (de 10 a 70, para la escala 1,0 a 7,0) |
| `int16` | el máximo no pasa de 32 767 | el número de pista de un álbum, la cantidad de estudiantes de un curso |
| `int32` | el máximo puede llegar a millones | la población de una comuna |
| `int64` | el máximo supera los 2 100 millones | una marca de tiempo en milisegundos desde 1970 (cerca de 1,7 × 10¹²) |
| `float32` | bastan unos 7 dígitos | la temperatura de un sensor, las variables de entrada de un modelo de aprendizaje automático, cuyo error suele ser mucho mayor que la diferencia entre `float32` y `float64` |
| `float64` | se necesitan más dígitos o se acumulan muchas operaciones | el total de millones de ventas, un cálculo científico cuyo resultado se compara con otro |
| entero en lugar de flotante | el valor debe ser exacto | un saldo en pesos chilenos, que no tiene decimales |

Dos fallas concretas: `np.array([200], dtype="uint8") + 100` devuelve `[44]`, porque 300 no cabe en un byte, y `np.float32(123456789)` devuelve `123456792.0`, porque con unos 7 dígitos no alcanzan 9 cifras.

**Cuándo importa el tipo y la codificación de un texto**

- Un código con ceros a la izquierda, como `04101` (la comuna de La Serena), se guarda como string: leído como número quedaría `4101`, y sumarlo o promediarlo no tiene sentido.
- Un número leído como string no se suma: `"1000" + "5"` da `"10005"`. Hay que convertirlo antes.
- Un archivo UTF-8 leído como `latin-1` muestra `Ã±` en vez de `ñ`, y uno `latin-1` leído como UTF-8 suele fallar con `UnicodeDecodeError`. Al leer un CSV con tildes, conviene indicar la codificación (`pd.read_csv(ruta, encoding="latin-1")`).
- Una columna de texto con pocos valores repetidos, como `genre` (10 géneros en 200 000 filas), no necesita guardar el mismo texto miles de veces: en pandas, el tipo `category` guarda un código entero por fila y una tabla con las etiquetas. Un texto largo y casi siempre distinto, como una letra, no se beneficia de esto.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición: cómo guarda Python los datos y por qué es ineficiente</strong>

En Python todo valor es un **objeto**: además del dato, lleva una cabecera con un contador de referencias, su tipo y, en los enteros, su tamaño. En CPython 3.13, `sys.getsizeof(1)` da 28 bytes, mientras que ese mismo entero ocuparía 8 en un `int64`; `sys.getsizeof(1.5)` da 24, y un `float64` ocupa 8. Los strings también son objetos, con su propia cabecera.

Una `list` de Python no guarda los valores sino **punteros** (8 bytes cada uno) a objetos que pueden estar en cualquier parte de la memoria. Un millón de enteros en una lista ocupa unos 34 MiB: 7,6 MiB de punteros y 26,7 MiB de objetos. Un arreglo de NumPy de tipo `int64` con los mismos valores ocupa 7,6 MiB, y uno `int32`, 3,8 MiB, porque guarda solo los valores, uno junto al otro y todos del mismo tipo.

<img src="assets/python-vs-numpy.svg" alt="Una lista de Python guarda punteros de 8 bytes hacia objetos de 28 bytes repartidos en la memoria; un arreglo int64 de NumPy guarda valores de 8 bytes contiguos." style="display:block; margin:12px auto; width:720px; max-width:100%; height:auto;">

Esa estructura tiene tres costos:

1. **Memoria:** cada dato carga una cabecera y un puntero, 36 bytes donde un `int64` necesita 8.
2. **Acceso:** los objetos no tienen por qué estar juntos, y el procesador, al leer un dato, trae también sus vecinos a la caché (una memoria pequeña y rápida). Con punteros esa ventaja se pierde.
3. **Operaciones:** en cada operación el intérprete comprueba el tipo de los operandos (`a + b` puede sumar enteros o concatenar strings), y como los enteros de Python crecen sin límite en vez de desbordar, cada suma debe revisar el tamaño del resultado en lugar de ejecutar una sola instrucción del procesador.

Por eso las columnas numéricas de NumPy, pandas y Polars guardan valores de un solo tipo fijo y contiguos, y cada valor ocupa solo los bytes de su tipo.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición: tipos numéricos, bits y rangos</strong>

Cada tipo numérico de NumPy y pandas fija cuántos bits ocupa un valor y, con ello, su rango. Elegir entre `int16` y `int32` exige comprobar el máximo de cada columna.

| Tipo | Tamaño | Rango |
| --- | --- | --- |
| `int16` | 2 bytes | −32 768 a 32 767 |
| `int32` | 4 bytes | −2 147 483 648 a 2 147 483 647 |
| `int64` | 8 bytes | aprox. ±9,2 × 10¹⁸ |

Entre los flotantes, `float32` ocupa 4 bytes y `float64`, 8.

`np.array([32767], dtype="int16") + 1` devuelve `[-32768]`: no hay error ni aviso, el valor da la vuelta al otro extremo del rango. En este dataset, `duration_ms` llega a 1 047 307 y no cabe en `int16`; `total_artist_followers` llega a unos 295 millones y sí cabe en `int32`.

Para medir memoria, `memory_usage` entrega bytes. El código de más abajo divide por `1024**2`, así que sus resultados están en MiB, según las unidades definidas al comienzo.
</div>

**Código dado** — funciones de carga:

In [ ]:
import pandas as pd


def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(
    data_dir: Path, n_batches: int | None = None
) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

<!-- EJERCICIO -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #06b6d4; background:rgba(6,182,212,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💻 Ejercicio 1.1: reducir la memoria del DataFrame (0,3 puntos)</strong>
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a una copia del DataFrame cargado y guardarla en `df_opt`:
  - `float64` → `float32`: `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo` y `avg_artist_popularity`.
  - `int64` → `int16`: `key` y `mode`.
  - `int64` → `int32`: `year`, `popularity`, `duration_ms` y `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

Trabajen en la celda siguiente y guarden el DataFrame cargado en `df`. `df_opt` debe conservar las 200.000 filas y las 24 columnas de `df`, con menos memoria; el gráfico de la celda que viene después (código dado) muestra ambos usos en su título.
</div>

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan df y df_opt antes de continuar.")

**Código dado:** comparación del uso de memoria antes y después.

In [ ]:
import plotly.express as px

mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

<!-- EJERCICIO -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #06b6d4; background:rgba(6,182,212,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💻 Ejercicio 1.1 (parte 2): leer solo las columnas necesarias (0,2 puntos)</strong>
- [ ] Definir la lista de columnas que usa el modelo de la sección 2: `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `tempo`, `duration_ms`, `year`, `key`, `mode`, `genre` y la variable que se predice, `valence`.
- [ ] Leer los 20 archivos pidiendo solo esas columnas y guardar el resultado en `df_cols` (Hint: el parámetro `columns` de `pd.read_parquet`).
- [ ] Comparar el uso de memoria de `df`, `df_opt` y `df_cols` (código dado).

Trabajen en la celda siguiente. `df_cols` debe tener las 200.000 filas de `df` y 14 columnas.
</div>

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan df_cols antes de continuar.")

**Código dado:** comparación del uso de memoria con las tres lecturas.

In [ ]:
mem_cols = df_cols.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Original", "Tipos ajustados", "Solo las columnas necesarias"],
    y=[mem_before, mem_after, mem_cols],
    labels={"x": "Lectura", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} → {mem_after:.1f} → {mem_cols:.1f} MiB",
).show()

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (0,5 puntos)</strong>
  1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
  2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué se gana al usar `pd.read_parquet` en vez de `pd.read_csv`?
  3. Un colega propone guardar `duration_ms` y `total_artist_followers` como `float32` para ahorrar memoria. Comprueben sus máximos y conviértanlos con `np.float32(...)`: ¿en cuál de las dos columnas aparecería un problema y por qué?
  4. Un banco guarda saldos en pesos como `float32`. ¿Qué falla concreta aparecería y qué tipo usarían en su lugar?
  5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (mencionen al menos dos)
  6. ¿Cuánto se redujo el uso de memoria con la conversión de tipos y cuánto leyendo solo las columnas necesarias (en MiB y en %)? ¿Por qué una de las dos técnicas ahorra mucho más que la otra?
  7. ¿Qué pasaría si intentáramos reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?
  8. `year` y `popularity` caben en `int16`, pero el ejercicio pide `int32`. Comprueben sus rangos con `agg(["min", "max"])`. ¿Qué criterio usarían para elegir entre ambos tipos y cuánta memoria cambiaría esa elección?
</div>

`Escriban sus respuestas aquí:`

In [ ]:
# Una vez contestadas las preguntas, ejecuten esta celda para liberar memoria.
df_opt = None
df_cols = None

### 1.2 Lectura con Hilos y Procesos (1,0 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Listo, pero ¿cuánto se demora?</strong>

Con el catálogo ya en un tamaño razonable, Toño logra abrirlo sin que el computador se trabe, y Pía puede preparar su programa en paz. Hernán casi se emociona, pero queda un detalle: cada mañana hay que leer los 20 archivos uno tras otro, y en verano llegarán canciones nuevas que obligarán a recargar el catálogo seguido. Toño sospecha que se puede leer más de un archivo a la vez, y de inmediato pone una condición: «Si sale más rápido pero me cambian las canciones, prefiero esperar». Hernán está de acuerdo. La lectura en paralelo solo le sirve si el catálogo queda idéntico y si pueden mostrarle cuánto se gana de verdad.
</div>


<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Núcleos físicos y procesadores lógicos</strong>

Un **núcleo físico** (*core*) es una unidad de ejecución de la CPU. Varios núcleos permiten ejecutar trabajo simultáneamente. Algunos procesadores permiten que cada núcleo mantenga más de un hilo de hardware, técnica llamada **SMT**: el sistema operativo ve entonces varios **procesadores lógicos** por núcleo.

Un computador de 6 núcleos y 12 procesadores lógicos sigue teniendo 6 núcleos físicos: cada pareja de procesadores lógicos comparte recursos del núcleo. Esa organización puede aprovechar mejor los recursos, pero no equivale a duplicarlos ni garantiza el doble de rendimiento. Los hilos de software que creen los programas se asignan a esos recursos; puede haber muchos más hilos de software que procesadores lógicos.
<img src="assets/nucleos-fisicos-logicos.svg" alt="Ejemplo de dos núcleos físicos con cuatro procesadores lógicos SMT. Los hilos de software son planificados sobre recursos que se comparten dentro de cada núcleo." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Hilos y procesos: qué comparten los trabajadores</strong>

Un **programa** es el código que podemos ejecutar. Un **proceso** es una instancia en ejecución, con su propio espacio de memoria y recursos. Dentro de un proceso puede haber varios **hilos de software** (*threads*): cada uno recorre una secuencia de instrucciones y comparte la memoria del proceso con los demás. Crear otro hilo no crea otro núcleo físico.

Un **pool** mantiene un conjunto de trabajadores y les asigna tareas. La diferencia entre ambos ejecutores afecta qué pueden compartir y cuánto cuesta devolver el resultado.

| Aspecto | `ThreadPoolExecutor` | `ProcessPoolExecutor` |
|---|---|---|
| Trabajadores | Hilos de un proceso | Procesos con intérpretes separados |
| Memoria | Comparten los objetos del proceso; modificar el mismo objeto exige coordinación | Tienen espacios de memoria separados; modificar una copia no modifica el objeto del padre |
| Argumentos y resultados | Se pasan referencias dentro del proceso | Deben ser serializables: se codifican para enviarlos y se reconstruyen al recibirlos |
| Código Python intensivo en CPU | Con el GIL activo, los hilos no ejecutan bytecode Python simultáneamente | Los procesos pueden ejecutar bytecode simultáneamente en varios núcleos |
| Costos | Crear hilos, asignar tareas y coordinar accesos | Arrancar procesos, importar librerías, transferir datos y coordinar tareas |

Usamos CPython con el GIL activo. Las extensiones nativas pueden liberarlo: por eso un trabajo con mucho cómputo, como decodificar Parquet en Arrow, también puede beneficiarse de los hilos. La etiqueta «CPU-bound» por sí sola no decide el ejecutor.

En la comparación con `load_all_process` que ejecutaremos en esta sección, cada proceso recibirá una ruta y devolverá un DataFrame: enviar la ruta cuesta poco, pero devolver la tabla requiere transferir sus datos. Además, una función definida en el notebook puede no ser importable por un proceso nuevo; por eso se entrega `pd.read_parquet`, que pertenece a un módulo importable. Ver [documentación de concurrent.futures](https://docs.python.org/3.13/library/concurrent.futures.html).
<img src="assets/hilos-procesos-memoria.svg" alt="Dos hilos pueden acceder a un objeto en la memoria de un proceso; procesos separados envían y reconstruyen datos en sus propios espacios de memoria." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Secuencial, concurrente y paralelo</strong>

En ejecución **secuencial**, una tarea termina antes de comenzar la siguiente. Hay **concurrencia** cuando varias tareas avanzan durante un mismo intervalo: pueden turnarse en un solo núcleo mientras esperan datos. Hay **paralelismo** cuando varias ejecutan trabajo simultáneamente en recursos distintos, por ejemplo en dos núcleos.

Tener varios hilos permite organizar tareas concurrentes, pero no demuestra que estén calculando a la vez. Más adelante veremos async, donde una tarea puede ceder el control durante una espera, y compararemos esa coordinación con el trabajo que ejecutan hilos y procesos.
<img src="assets/secuencial-concurrente-paralelo.svg" alt="Las mismas tareas se ejecutan una después de otra, intercalando cálculos en un núcleo o calculando en dos núcleos. Los segmentos de espera se distinguen de los de cálculo." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Sincronización y cantidad de trabajadores</strong>

La **sincronización** coordina tareas para que observen resultados completos o accedan de forma segura a datos compartidos. Una **condición de carrera** ocurre cuando el resultado depende del orden en que se intercalan sus operaciones. Si dos hilos leen un contador en 5 y ambos escriben 6, una de las dos actualizaciones se pierde; el resultado esperado era 7. Un bloqueo de exclusión mutua puede proteger toda la operación de leer, modificar y escribir. El GIL no convierte una secuencia de operaciones en una actualización indivisible.

<img src="assets/condicion-carrera.svg" alt="Dos hilos leen un contador en cinco y escriben seis; una actualización se pierde. La protección debe abarcar lectura, modificación y escritura." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

En la lectura del catálogo, cada trabajador construye su propia tabla; el hilo principal reúne los resultados al terminar. Así evitamos que varios trabajadores escriban sobre el mismo DataFrame durante la lectura.

La **saturación** aparece cuando agregar trabajadores ya no aumenta el trabajo completado por segundo: pueden estar compitiendo por el mismo disco, memoria o CPU. Crear más trabajadores que recursos útiles también agrega turnos y coordinación. Además, algunas bibliotecas crean sus propios hilos: varias tareas externas pueden multiplicar el número total. Por eso `max_workers` y `n_jobs` controlan trabajadores, no prometen una aceleración del mismo tamaño.
</div>

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al solapar la espera de I/O (disco/red) con el trabajo del procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Benchmark, perfilado y tiempos</strong>

Un **benchmark** mide el costo de un trabajo definido para comparar alternativas: debemos mantener datos y resultado esperado, y declarar qué entra en el tiempo. El **perfilado** (*profiling*) descompone una ejecución por funciones para localizar dónde se consume ese tiempo. Primero mediremos la lectura completa; después usaremos un perfil para investigar sus costos.

El **calentamiento** (*warm-up*) ejecuta trabajo antes de medir el comportamiento posterior al arranque. Puede dejar archivos en la caché del sistema operativo, inicializar un pool o compilar código; son efectos distintos. Calentar una función no elimina por sí solo todos los demás costos. Si importa la primera solicitud, su tiempo se informa aparte en vez de descartarlo.

Las mediciones varían por otros programas, cachés y decisiones del sistema operativo. Por eso repetimos. La **media** es la suma de los tiempos dividida por la cantidad de mediciones; la **mediana** es el valor central al ordenarlos y resulta menos sensible a una ejecución aislada muy lenta. La **desviación estándar** describe cuánto se dispersan los tiempos respecto de su media. Una mediana no muestra toda esa variación, ni repetir reemplaza controlar qué se compara.

- `%timeit instrucción` (en Jupyter) repite la instrucción en varias rondas y entrega la media ± la desviación estándar de las rondas. Ajusta solo el número de repeticiones según lo que demora.
- `time.perf_counter()` devuelve un reloj de alta resolución, en segundos. El tiempo de un bloque es el valor tomado después menos el tomado antes. Una sola medición varía entre ejecuciones: los benchmarks de 2.1 y 2.2 no la repiten, así que sus gráficos pueden tener ruido.
- **Speedup:** tiempo de la versión base dividido por el tiempo de la versión mejorada. Un speedup de 2 significa que la versión nueva tarda la mitad; uno menor que 1 significa que es más lenta que la base.
- **Overhead:** tiempo adicional que no es el trabajo útil, sino lo necesario para poder hacerlo. Por ejemplo, abrir una conexión a una base de datos suma tiempo a la consulta misma.
</div>

<!-- EJERCICIO -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #06b6d4; background:rgba(6,182,212,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💻 Ejercicio 1.2: lectura en paralelo (0,5 puntos)</strong>
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Ejecutar el benchmark de más abajo (código dado), que mide los tiempos para 2, 3, 4, …, 20 archivos y grafica las series `Serial` y `Paralelo`, e interpretar el gráfico en las preguntas.
- [ ] Ejecutar las tres celdas de código dado que siguen al benchmark (CPU frente a reloj, perfil de `load_all_serial` y comparación con procesos) y usar sus resultados en las preguntas 4, 7 y 8.

Implementen la función en la primera celda de abajo, con la misma firma que `load_all_serial`: con los 20 batches debe devolver las mismas 200.000 filas y 24 columnas que la versión en serie. Los `%timeit` van en la segunda celda.
</div>

<!-- TIP -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47,129,247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Herramientas para `load_all_parallel`</strong>

Investiguen `ThreadPoolExecutor` usado con `with`, su método `map` y el parámetro `max_workers`. `load_batch` ya lee un archivo; revisen `load_all_serial` para ver cómo se arma el DataFrame final. Antes de medir, comprueben que el resultado tiene las mismas 200.000 filas y en el mismo orden que el de `load_all_serial`.
</div>

In [ ]:
def load_all_parallel(
    data_dir: Path, n_batches: int | None = None
) -> pd.DataFrame:
    raise NotImplementedError("Completen load_all_parallel antes de ejecutar esta celda.")

In [ ]:
# Su código aquí


**Código dado:** benchmark que mide los tiempos de lectura para 2, 3, …, 20 archivos y los grafica. Alterna quién corre primero en cada iteración, porque el sistema operativo guarda en caché los archivos recién leídos y favorecería a la versión que corre después.

In [ ]:
import time
from dataclasses import dataclass


@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

def medir(version, funcion, n):
    t0 = time.perf_counter()
    funcion(RAW_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, version))


for n in range(2, 21):
    orden = [("Serial", load_all_serial), ("Paralelo", load_all_parallel)]
    if n % 2 == 0:
        orden.reverse()
    for version, funcion in orden:
        medir(version, funcion, n)

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

**Código dado:** compara el tiempo de CPU con el tiempo de reloj al leer los 20 archivos. `time.process_time()` suma el tiempo de CPU de todos los hilos del proceso. La razón CPU / reloj expresa el consumo medio agregado: cerca de 1 equivale, en promedio, al consumo de un núcleo; mayor que 1 muestra que hubo ejecución simultánea en varios núcleos durante parte del intervalo. Una razón menor que 1 indica un consumo medio inferior al de un núcleo ocupado, pero no identifica la causa. El promedio puede combinar cálculo paralelo y esperas: contrástenlo con el perfil y el estado de la caché antes de concluir qué limita la lectura.

In [ ]:
def medir_cpu(funcion):
    t0, c0 = time.perf_counter(), time.process_time()
    funcion(RAW_DIR)
    reloj, cpu = time.perf_counter() - t0, time.process_time() - c0
    print(
        f"{funcion.__name__}: reloj {reloj:.2f} s | CPU {cpu:.2f} s | "
        f"CPU / reloj = {cpu / reloj:.1f}"
    )


medir_cpu(load_all_serial)
medir_cpu(load_all_parallel)

**Código dado:** perfil de `load_all_serial`. La columna `cumtime` es el tiempo acumulado de cada función, incluidas las que ella llama. Miren `read_table` y `arrow_table_to_pandas`.

In [ ]:
import cProfile
import pstats

perfil = cProfile.Profile()
perfil.enable()
load_all_serial(RAW_DIR)
perfil.disable()
pstats.Stats(perfil).sort_stats("cumulative").print_stats(10)

**Código dado:** compara la lectura en serie, con threads y con procesos (`ProcessPoolExecutor`). Cada medición se repite 3 veces y se informa la mediana.

In [ ]:
import statistics
from concurrent.futures import ProcessPoolExecutor


def load_all_process(
    data_dir: Path, n_batches: int | None = None
) -> pd.DataFrame:
    """Lee los archivos Parquet de data_dir en procesos separados y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    # Se usa pd.read_parquet y no load_batch: en Windows y macOS los procesos
    # nuevos no ven las funciones definidas en el notebook.
    with ProcessPoolExecutor() as executor:
        dfs = list(executor.map(pd.read_parquet, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


for funcion in (load_all_serial, load_all_parallel, load_all_process):
    tiempos = []
    for _ in range(3):
        t0 = time.perf_counter()
        funcion(RAW_DIR)
        tiempos.append(time.perf_counter() - t0)
    print(f"{funcion.__name__}: mediana {statistics.median(tiempos):.2f} s")

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (0,5 puntos)</strong>
  1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
  2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
  3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
  4. ¿Cómo cambian el acceso a un DataFrame y la transferencia de resultados entre `ThreadPoolExecutor` y `ProcessPoolExecutor`? ¿Qué muestran las medianas de la celda con `load_all_process` y por qué ordenan así las tres lecturas? ¿Cuál usarían si el cálculo estuviera escrito en Python puro y mantuviera el GIL ocupado?
  5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
  6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
  7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan? Apóyense en el perfil de la lectura en serie.
  8. Con la razón CPU / reloj que imprime `medir_cpu`, ¿la lectura de estos archivos pasa la mayor parte del tiempo esperando disco o calculando? ¿Qué cambiaría si los archivos no estuvieran en la caché del sistema?
</div>

`Escriban sus respuestas aquí:`

### 1.3 Async Local (0,5 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Pía no quiere dejar a los oyentes esperando</strong>

Con la comparación de lectores, Toño ya tiene cómo justificar la carga del catálogo. Pía necesita resolver otra espera: antes de cerrar la programación de Radio Faro Papaya, debe consultar en qué franja quedaron confirmadas las solicitudes de cuatro oyentes. El servicio devuelve el identificador de cada solicitud y su franja, pero cada respuesta tarda un poco. Si espera una respuesta antes de enviar la siguiente consulta, esas esperas se acumulan.

«Si la primera se demora, que las otras no tengan que esperar también», les pide Pía. Ustedes probarán otra forma de organizar las consultas, conservando todas las respuestas y su orden. Para ensayar sin depender de la conexión de la playa, usarán un servicio simulado en el mismo computador.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Concurrencia cooperativa</strong>

Cuando una consulta espera una respuesta, otra podría avanzar durante esa espera. Para organizar ese trabajo usaremos **corrutinas**. Una **función de corrutina** se declara con `async def`; al llamarla se crea una corrutina, pero esa llamada por sí sola no ejecuta su cuerpo. Debemos esperar su ejecución con `await` o programarla como tarea.

El **event loop** coordina las tareas. Cuando una llega a un `await` que la suspende, el loop puede avanzar otra. Por eso hablamos de concurrencia **cooperativa**: las tareas deben ceder el control. El código Python que sigue calculando sin suspenderse continúa ocupando el hilo del loop.

En el ensayo usaremos `asyncio.sleep` para representar la espera de un servicio; no habrá solicitudes de red. Si esperamos cada consulta dentro de un `for` antes de iniciar la siguiente, las consultas avanzan de una en una. `asyncio.gather` permite programar varias corrutinas concurrentemente y devuelve sus resultados en el orden de los argumentos.

Una llamada bloqueante como `time.sleep` o `pd.read_parquet`, ejecutada directamente dentro de la corrutina, bloquea el hilo del loop: durante ese tiempo las otras tareas tampoco avanzan. `asyncio.to_thread` permite delegar una operación bloqueante a un hilo, que será quien la ejecute.

En el notebook usamos `await` directamente porque el kernel ya dispone de un event loop. Ver [documentación de asyncio](https://docs.python.org/3.13/library/asyncio-task.html).
</div>

**Código dado:** servicio simulado y consulta secuencial. Cada respuesta contiene el identificador de la solicitud y la franja confirmada; las latencias son datos del ensayo. Antes de ejecutar, anticipen el tiempo secuencial y el concurrente a partir de las cuatro esperas.

In [ ]:
import asyncio
import time

SOLICITUDES = {
    "oyente_01": (0.1, "mañana"),
    "oyente_02": (0.2, "tarde"),
    "oyente_03": (0.1, "noche"),
    "oyente_04": (0.2, "mañana"),
}


async def consultar_solicitud(solicitud: str) -> tuple[str, str]:
    latencia, franja = SOLICITUDES[solicitud]
    await asyncio.sleep(latencia)
    return solicitud, franja


async def consultar_en_serie(solicitudes: list[str]) -> list[tuple[str, str]]:
    respuestas = []
    for solicitud in solicitudes:
        respuestas.append(await consultar_solicitud(solicitud))
    return respuestas


solicitudes = list(SOLICITUDES)
inicio = time.perf_counter()
respuestas_serie = await consultar_en_serie(solicitudes)
tiempo_serie = time.perf_counter() - inicio
print(f"Secuencial: {tiempo_serie:.3f} s; respuestas: {respuestas_serie}")

<!-- EJERCICIO -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #06b6d4; background:rgba(6,182,212,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💻 Ejercicio 1.3: consultas concurrentes (0,2 puntos)</strong>

La consulta secuencial ya les permite observar qué devuelve el servicio y cuánto se acumulan las esperas. Ahora deben consultar las mismas solicitudes de forma concurrente, manteniendo el contrato de respuesta para que Pía pueda usar el resultado sin reorganizarlo.

- [ ] Implementar `consultar_concurrente` en la celda siguiente usando `asyncio.gather` y `consultar_solicitud`, sin cambiar las latencias.
- [ ] Devolver una lista con una respuesta por solicitud, en el orden de `solicitudes`, sin omisiones ni repeticiones. Para una lista vacía, devolver una lista vacía.
- [ ] Ejecutar la comparación provista y comprobar que el resultado coincide con `respuestas_serie`. Explicar los tiempos en las preguntas siguientes. No se exige una aceleración fija: esta es una simulación de espera, no un benchmark de red.
</div>

In [ ]:
async def consultar_concurrente(
    solicitudes: list[str],
) -> list[tuple[str, str]]:
    raise NotImplementedError(
        "Completen consultar_concurrente antes de ejecutar esta celda."
    )

**Código dado:** comparación y comprobación de respuestas. Se verifica también que una lista vacía no produzca solicitudes.

In [ ]:
inicio = time.perf_counter()
respuestas_concurrentes = await consultar_concurrente(solicitudes)
tiempo_concurrente = time.perf_counter() - inicio
assert respuestas_concurrentes == respuestas_serie
assert await consultar_concurrente([]) == []
print(f"Concurrente: {tiempo_concurrente:.3f} s")
print(f"Speedup del ensayo: {tiempo_serie / tiempo_concurrente:.2f}×")
print("Mismas cuatro respuestas, en el mismo orden.")

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (0,3 puntos)</strong>

1. Comparen los tiempos observados con la suma y el máximo de las latencias. ¿Por qué cada versión se aproxima a uno de esos valores y qué costos quedan fuera del ensayo?
2. Si el servicio usara `time.sleep` en lugar de `await asyncio.sleep`, ¿qué pasaría con las otras consultas y por qué `async def` no lo evitaría?
3. Si las solicitudes requirieran un cálculo largo escrito en Python puro, ¿qué cambiarían para aprovechar varios núcleos: async, hilos o procesos? Justifiquen usando el GIL y el costo de transferir datos.
</div>

`Escriban sus respuestas aquí:`

## 2. Predicción de Valencia

La columna `valence` de Spotify mide la **positividad musical**: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

### 2.1 Regresión Lineal a Mano (1,5 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Una canción nueva, un puntaje</strong>

Con el catálogo cargado, Pía llega con una petición: cada vez que un oyente done una canción, quiere saber en segundos qué tan alegre es para ubicarla en la franja correcta. Hernán ya tiene un modelo lineal que estima `valence` a partir de las características de audio, con sus coeficientes calculados. Lo que no sabe es cómo conviene programarlo. Toño tiene cinco opiniones, una por cada tutorial que vio (Python puro, NumPy, Numba, pandas y Polars), y todas aseguran ser la mejor. Como la radio tiene un solo computador y la transmisión no puede esperar a que termine una cuenta, Hernán no quiere elegir por fama. Les pide que midan cada una y le digan cuál usar.
</div>

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado** — carga de datos y parámetros del modelo:

In [ ]:
from sklearn.model_selection import train_test_split

df_train = load_all_serial(RAW_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[[*PARAM_COLS, "key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import numpy as np

params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analicen cómo cada implementación aborda el mismo problema y presten atención a las diferencias en legibilidad, concisión y (como verán en el benchmark) rendimiento.

In [ ]:
import numba
import polars as pl


def linear_regression_predict(
    X: np.ndarray, params: list[float]
) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(
    X: np.ndarray, params: list[float]
) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(
    X: np.ndarray, params: np.ndarray
) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(
    X: pd.DataFrame, params: list[float]
) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(
    X: pl.DataFrame, params: list[float]
) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones en modo eager)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Leer los gráficos del benchmark</strong>

En una **escala lineal**, distancias iguales representan diferencias iguales: de 1 a 2 se avanza lo mismo que de 2 a 3. En una **escala logarítmica**, distancias iguales representan razones iguales: de 0,001 a 0,01 se avanza lo mismo que de 0,01 a 0,1, porque en ambos casos multiplicamos por 10. Esta escala permite comparar tiempos de órdenes de magnitud distintos; no representa cero ni valores negativos.

El benchmark siguiente muestra los tiempos en escala lineal y luego con el eje vertical logarítmico. En ese segundo gráfico, una misma separación vertical indica la misma proporción entre tiempos, aunque la diferencia en segundos cambie. Cada tamaño e implementación se mide una sola vez; el calentamiento explícito es solo para Numba. Interpreten las curvas como mediciones con posible variación, no como una clasificación definitiva para cualquier ejecución.
<img src="assets/escala-lineal-logaritmica.svg" alt="Distancias iguales suman uno en la escala lineal y multiplican por diez en la logarítmica. Son ejemplos de escalas, no mediciones." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

In [ ]:
import time
from dataclasses import dataclass


@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

# Warm-up: la primera llamada a Numba compila la función; se hace aquí, con
# pocas filas, para que esa compilación no entre en las mediciones.
linear_regression_predict_numba(
    X_test[PARAM_COLS].iloc[:10].to_numpy(dtype=np.float32), params_arr
)

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(
            TimeMeasurement(time.perf_counter() - t0, it, name)
        )

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

<!-- COMPROBACIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación: las cinco implementaciones dan lo mismo</strong>

Antes de recomendar una implementación a Hernán, comprueben que las cinco entregan las mismas predicciones. Ejecuten la celda siguiente (código dado): la diferencia máxima con NumPy debe ser menor que 0,0001 en cada implementación.
</div>

In [ ]:
X_chk = X_test[PARAM_COLS].iloc[:1000]
X_chk_np = X_chk.to_numpy(dtype=np.float32)
referencia = np.asarray(linear_regression_predict_numpy(X_chk_np, params_vals))

candidatas = {
    "Python": np.array(linear_regression_predict(X_chk_np, params_vals)),
    "Numba-JIT": linear_regression_predict_numba(X_chk_np, params_arr),
    "Pandas": linear_regression_predict_pandas(X_chk, params_vals).to_numpy(),
    "Polars": linear_regression_predict_polars(
        pl.from_pandas(X_chk), params_vals
    ).to_numpy(),
}
for nombre, pred in candidatas.items():
    diferencia = np.abs(pred - referencia).max()
    print(f"{nombre}: diferencia máxima con NumPy = {diferencia:.2e}")
    assert diferencia < 1e-4, f"{nombre} no coincide con NumPy"

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (1,5 puntos)</strong>
  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador `@numba.njit`? ¿Qué significa el modo `nopython`?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  5. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  6. ¿Cuál implementación fue la más rápida en su medición? ¿Era esperable ese resultado?
  7. ¿Polars fue más eficiente que pandas en su medición? Verifiquen la versión de pandas instalada (`pd.__version__`) y comenten si creen que la versión influye en el resultado.
  8. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeran ese costo? ¿En qué escenarios de producción ese costo no existiría?
</div>

`Escriban sus respuestas aquí:`

### 2.2 Entrenamiento y Comparación de `n_jobs` (1,0 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 El verano exige más</strong>

Un oyente le prestó a Hernán, para toda la temporada, un computador con varios núcleos. Con eso, Hernán sueña con un modelo más fino que el lineal: un `RandomForestRegressor` que, además del audio, tome en cuenta el género de la canción. Antes de apoyarse en él quiere resolver tres dudas. La primera es si de verdad mejora al modelo lineal y a la opción más simple de todas, predecir siempre el promedio. La segunda es cuánto ayuda usar todos los núcleos para entrenarlo. La tercera le importa más: si el resultado cambia cuando cambia la configuración. «Si hoy el modelo dice que una canción es alegre, mañana no puede decir lo contrario solo porque entrenamos de otra forma», les advierte. Tienen que comprobarlo con una métrica.
</div>

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        (
                            "ohe",
                            OneHotEncoder(handle_unknown="ignore"),
                            ["key", "mode", "genre"],
                        ),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            (
                "random_forest",
                RandomForestRegressor(n_jobs=n_jobs, random_state=42),
            ),
        ]
    )

In [ ]:
import time

from sklearn.metrics import root_mean_squared_error

# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

In [ ]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

**Código dado:** RMSE de dos referencias sobre el mismo conjunto de prueba: predecir siempre el promedio de `valence` del entrenamiento y el modelo lineal de la sección 2.1.

In [ ]:
rmse_media = root_mean_squared_error(
    y_test, np.full(len(y_test), y_train.mean())
)
pred_lineal = X_test[PARAM_COLS].to_numpy() @ np.array(params_vals) + intercept
rmse_lineal = root_mean_squared_error(y_test, pred_lineal)

print(f"Predecir el promedio → RMSE: {rmse_media:.4f}")
print(f"Regresión lineal     → RMSE: {rmse_lineal:.4f}")
print(f"Random Forest        → RMSE: {rmse_all:.4f}")

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (1,0 puntos)</strong>
  1. Antes de interpretar los tiempos, predigan qué cambia entre `n_jobs=1`, `n_jobs=-1` y `n_jobs=k` en la cantidad de trabajadores que puede usar el bosque. Si `k` supera los procesadores lógicos disponibles, ¿por qué eso no implica disponer de más núcleos ni garantiza menor tiempo?
  2. ¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?
  3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`?
  4. ¿Fue proporcional al número de procesadores lógicos disponibles en su máquina? ¿Por qué no?
  5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?
  6. ¿El bosque mejora al modelo lineal y a predecir el promedio? ¿Cuánto? ¿Compensa el error que se reduce los segundos que tarda en entrenarse con `n_jobs=-1`?
</div>

`Escriban sus respuestas aquí:`

### 2.3 Decisiones sobre GPU (0,5 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 La oferta del oyente gamer</strong>

Hernán ya cuenta con los tiempos y errores de los modelos para decidir cómo incorporar las predicciones a Radio Faro Papaya. Entonces aparece una oferta: un oyente podría prestar una GPU durante el verano. Toño empieza a hablar de predicciones instantáneas, pero Pía le recuerda que a veces necesita clasificar una sola canción y otras prepara la programación de toda la semana.

Hernán quiere saber si el préstamo ayudaría en ambos casos. Que el cálculo sea rápido le sirve de poco si preparar los datos y recuperar el resultado consume lo que esperaba ahorrar. Antes de que Toño anuncie el préstamo al aire, les pide comparar esos costos y explicar en qué condiciones convendría usar la GPU.
</div>

Para analizar esta propuesta, pueden medir una GPU compatible o usar la referencia docente provista. Distingan esa referencia de sus mediciones propias y de los datos hipotéticos de las preguntas.

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 GPU: muchas operaciones sobre muchos datos</strong>

La **GPU** (*Graphics Processing Unit*, unidad de procesamiento gráfico) se diseñó para procesar gráficos y también puede ejecutar cálculos numéricos. Agrupa muchas unidades de ejecución para aplicar operaciones a grandes conjuntos de datos. Una CPU también realiza trabajo paralelo, pero dedica recursos a ejecutar flujos de instrucciones variados; una GPU suele aprovechar mejor lotes grandes con operaciones similares.

Las unidades que un fabricante llama «núcleos GPU» no equivalen a núcleos CPU: contar unos y otros no permite comparar su rendimiento. Si hay pocos datos o mucho costo de preparación y movimiento, la GPU puede tardar más.

Una GPU **dedicada** suele tener su propia memoria. Una **integrada**, como la de la referencia docente, comparte la memoria física del sistema con la CPU. Aun así, preparar arrays, coordinar accesos y copiar resultados puede tener costo: memoria compartida no significa operaciones gratuitas.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Host, dispositivo y software de ejecución</strong>

Llamaremos **host** al computador y al programa que coordina el trabajo en CPU; **dispositivo** al acelerador al que se envía el cálculo. Una entrada **residente** ya está preparada en el dispositivo antes de la operación medida. Según el hardware, colocarla allí puede requerir una transferencia a otra memoria o preparar su acceso en memoria compartida.

Una **biblioteca** ofrece funciones que usamos desde Python. Su **backend** es la implementación que ejecuta las operaciones para un recurso concreto, como CPU o GPU. El **controlador** (*driver*) es el software del sistema que permite comunicarse con el hardware. Un **complemento** (*plugin*) conecta componentes, por ejemplo JAX con el soporte de un fabricante. Instalar una biblioteca Python no instala necesariamente el controlador que requiere el dispositivo.

**CUDA**, **ROCm**, **oneAPI** y **Metal** son plataformas o conjuntos de herramientas para programar hardware; no son modelos de GPU. La ruta de instalación depende del fabricante, del dispositivo y del sistema operativo.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 JAX y la comparación entre dispositivos</strong>

**JAX** es una biblioteca Python para cálculo numérico con arrays, con una interfaz similar a NumPy. Puede ejecutar operaciones compatibles en CPU o aceleradores. `jax.jit` compila una función compatible mediante el compilador XLA, que puede optimizar conjuntamente varias operaciones y reducir resultados intermedios. La primera compilación tiene un costo distinto del de las ejecuciones posteriores.

En este laboratorio lo usamos para comparar la misma expresión en CPU y GPU. Comparar NumPy CPU con JAX GPU cambia tanto la implementación como el dispositivo; la curva **JAX CPU** ayuda a distinguir esos efectos. No asumiremos que toda diferencia respecto de NumPy sea mérito de la GPU. Ver [introducción de JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html).
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Tiempo de cómputo y tiempo total</strong>

Una GPU ejecuta muchas operaciones en paralelo, pero usarla también puede requerir transferir datos desde la RAM a su memoria y recuperar los resultados. Para el escenario de una operación aislada, sin solapamiento de etapas, usaremos:

`tiempo total GPU = transferencia de entrada + cómputo GPU + transferencia de salida`.

Si los datos permanecen en GPU entre operaciones, algunas transferencias se pagan una sola vez. Además, lanzar una operación puede devolver el control a la CPU antes de que termine el cómputo GPU: medir solo el lanzamiento subestima la duración. Un benchmark debe esperar la finalización y distinguir la primera ejecución de las siguientes. Ver [guía de rendimiento de CuPy](https://docs.cupy.dev/en/stable/user_guide/performance.html).

No necesitan instalar CuPy ni disponer de GPU para esta sección.
<img src="assets/gpu-residente-completa.svg" alt="GPU completa incluye preparar entradas, invocar, esperar y recuperar resultados. GPU residente parte con entradas listas e incluye la invocación y espera hasta finalizar." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

#### Instalación opcional de JAX con uv

Si tienen una GPU compatible, pueden medirla; si no, usen la referencia comentada que aparece antes del gráfico. Ambas rutas permiten analizar el costo de mover datos. Este bloque es código dado y no agrega puntaje.

Identifiquen su GPU y sistema operativo. Consulten la [guía de JAX](https://docs.jax.dev/en/latest/installation.html) para elegir el extra o complemento y comprobar requisitos de controladores, arquitectura y versiones. NVIDIA usa CUDA; AMD, ROCm; Intel, oneAPI; Apple Silicon, Metal. Las rutas Intel y Metal son experimentales. En Windows nativo estas celdas no instalan un backend GPU; consulten la compatibilidad con WSL2 o usen la referencia.

Completen `fabricante_gpu` con `nvidia`, `amd`, `intel` o `apple`; mantengan `ninguna` para omitir la instalación. Para NVIDIA, elijan `cuda12` o `cuda13` según la guía. Para AMD, averigüen el identificador de arquitectura, por ejemplo `gfx1103`: no usen ese ejemplo si no corresponde a su GPU. La ruta AMD fija la combinación JAX 0.11.0 / ROCm 10.0.0 probada para la referencia y requiere que su modelo esté soportado por ella.

<!-- WARNING -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Paquetes y controladores</strong>

`uv` instala los paquetes Python y sus bibliotecas, pero estas celdas no instalan controladores del sistema. Si falta un controlador o su combinación no está soportada, JAX puede fallar o mostrar solo CPU: en ese caso usen la referencia. Las descargas pueden ocupar varios GB. El entorno opcional se crea en su carpeta personal, separado del laboratorio; el notebook conserva su kernel.
</div>

In [ ]:
# Configuración de la instalación opcional.
import platform

instalar_gpu = False
fabricante_gpu = "ninguna"
extra_nvidia = "cuda13"
arquitectura_amd = ""
resultados_gpu = None
origen_gpu = "Mediciones propias"
print(f"Plataforma: {platform.system()} / {platform.machine()}")

In [ ]:
# Selección del paquete según fabricante y plataforma.
import re

sistema_gpu = platform.system()
maquina_gpu = platform.machine().lower()
paquetes_gpu = []
indice_gpu = None

if fabricante_gpu == "ninguna":
    print("Sin instalación GPU; pueden usar la referencia comentada.")
elif (fabricante_gpu == "nvidia" and sistema_gpu == "Linux"
      and maquina_gpu in {"x86_64", "aarch64"}):
    if extra_nvidia not in {"cuda12", "cuda13"}:
        raise ValueError("Elijan cuda12 o cuda13 según su GPU y controlador.")
    paquetes_gpu = [f"jax[{extra_nvidia}]"]
elif (fabricante_gpu == "amd" and sistema_gpu == "Linux"
      and maquina_gpu == "x86_64"):
    if not re.fullmatch(r"gfx[0-9a-f]+", arquitectura_amd):
        raise ValueError("Completen arquitectura_amd con el identificador gfx.")
    paquetes_gpu = [
        "jax==0.11.0",
        "jaxlib==0.11.0",
        "jax_rocm10_plugin==0.11.0+rocm10.0.0",
        "jax_rocm10_pjrt==0.11.0+rocm10.0.0",
        f"rocm[libraries,device-{arquitectura_amd}]==10.0.0",
    ]
    indice_gpu = "https://stable.repo.amd.com/rocm/whl-next/"
elif (fabricante_gpu == "intel" and sistema_gpu == "Linux"
      and maquina_gpu == "x86_64"):
    paquetes_gpu = ["jax[oneapi]"]
elif (fabricante_gpu == "apple" and sistema_gpu == "Darwin"
      and maquina_gpu == "arm64"):
    paquetes_gpu = ["jax-metal"]
else:
    print("No hay una instalación prevista para esa selección y plataforma.")
    print("Revisen la guía de compatibilidad o usen la referencia docente.")

print("Paquetes seleccionados:", paquetes_gpu)

In [ ]:
# Instalación en un entorno separado; solo corre con instalar_gpu = True.
import shutil
import subprocess
import sys
from pathlib import Path

if instalar_gpu and paquetes_gpu:
    uv_gpu = shutil.which("uv")
    if uv_gpu is None:
        raise RuntimeError("Instalen uv y reinicien VS Code para usarlo aquí.")
    entorno_gpu = (
        Path.home() / ".cache" / "ia7202" / f"jax-{fabricante_gpu}"
    )
    python_gpu = entorno_gpu / "bin" / "python"
    if not python_gpu.exists():
        subprocess.run(
            [uv_gpu, "venv", "--python", sys.executable, str(entorno_gpu)],
            check=True,
        )
    comando_gpu = [
        uv_gpu, "pip", "install", "--python", str(python_gpu), *paquetes_gpu
    ]
    if indice_gpu is not None:
        comando_gpu += ["--extra-index-url", indice_gpu]
    print("Instalando:", comando_gpu)
    subprocess.run(comando_gpu, check=True)
else:
    print("Instalación omitida. Para activarla, cambien instalar_gpu a True.")

#### Benchmark opcional en su GPU

El código siguiente mide la misma predicción sobre datos sintéticos uniformes entre 0 y 1, con diez características, semilla 42 y `float32`. Usamos los tamaños de lote de la referencia. Cada medición informa la mediana de quince repeticiones después de tres calentamientos; la compilación queda fuera del tiempo informado. La salida debe coincidir con NumPy dentro de `rtol=atol=1e-4`.

La **tolerancia absoluta** (`atol`) fija un margen de diferencia aun cerca de cero; la **relativa** (`rtol`) ajusta ese margen a la magnitud del valor de referencia. Con ambas en `1e-4`, se acepta cada resultado si cumple:

`abs(resultado - referencia) <= 1e-4 + 1e-4 * abs(referencia)`.

Por ejemplo, para una referencia de 2 el margen es 0,0003; para una referencia de 0, es 0,0001. Estas tolerancias comprueban coincidencia numérica en el ensayo; no demuestran por sí solas que cambiar la precisión sea adecuado para cualquier uso de las predicciones.

**GPU residente** mide el cálculo con entradas ya preparadas en el dispositivo. **GPU completa** incluye colocar las entradas, calcular y copiar la salida a NumPy; no incluye lectura de archivos ni arranque del intérprete. Cada llamada espera el resultado con `block_until_ready()`, porque lanzar una operación no significa que ya terminó. Ver [benchmarking de JAX](https://docs.jax.dev/en/latest/benchmarking.html).

Cambien `medir_gpu` a `True` después de instalar. La prueba se ejecuta con el Python del entorno opcional y comprueba que haya un dispositivo acelerador antes de medir. Si usan la referencia, dejen esta opción en `False`.

In [ ]:
# Medición opcional en el entorno instalado.
import json

medir_gpu = False

codigo_benchmark_gpu = r"""
import json
import statistics
import time
import jax
import numpy as np

aceleradores = []
for dispositivo in jax.devices():
    if dispositivo.platform != "cpu":
        aceleradores.append(dispositivo)
if not aceleradores:
    raise RuntimeError("JAX solo detectó CPU; usen la referencia docente.")
gpu = aceleradores[0]
cpu = jax.devices("cpu")[0]
rng = np.random.default_rng(42)


def predecir(x, pesos):
    return (x * pesos).sum(axis=1) + np.float32(6.9481548)


def medir(funcion):
    for _ in range(3):
        funcion()
    tiempos = []
    for _ in range(15):
        inicio = time.perf_counter()
        funcion()
        tiempos.append((time.perf_counter() - inicio) * 1000)
    return statistics.median(tiempos)


resultados = []
for filas in (1, 1000, 40000, 200000, 1000000):
    x = rng.random((filas, 10), dtype=np.float32)
    pesos = rng.random(10, dtype=np.float32)
    referencia = predecir(x, pesos)
    registro = {"filas": filas,
                "NumPy CPU": medir(lambda: predecir(x, pesos))}
    compilada = jax.jit(predecir)
    for nombre, dispositivo in (("JAX CPU", cpu), ("GPU residente", gpu)):
        x_dispositivo = jax.device_put(x, dispositivo)
        pesos_dispositivo = jax.device_put(pesos, dispositivo)
        jax.block_until_ready((x_dispositivo, pesos_dispositivo))
        salida = compilada(x_dispositivo, pesos_dispositivo)
        salida.block_until_ready()
        assert salida.device == dispositivo
        np.testing.assert_allclose(
            np.asarray(salida), referencia, rtol=1e-4, atol=1e-4
        )
        registro[nombre] = medir(
            lambda: compilada(x_dispositivo, pesos_dispositivo)
                    .block_until_ready()
        )

    def completa():
        x_gpu = jax.device_put(x, gpu)
        pesos_gpu = jax.device_put(pesos, gpu)
        salida = compilada(x_gpu, pesos_gpu).block_until_ready()
        return np.array(salida, copy=True)

    registro["GPU completa"] = medir(completa)
    resultados.append(registro)
print(json.dumps({"gpu": gpu.device_kind, "jax": jax.__version__,
                  "numpy": np.__version__, "resultados": resultados}))
"""

if medir_gpu:
    if not (instalar_gpu and paquetes_gpu):
        raise RuntimeError("Activen primero la instalación de su GPU.")
    ejecucion_gpu = subprocess.run(
        [str(python_gpu), "-c", codigo_benchmark_gpu],
        capture_output=True, text=True,
    )
    if ejecucion_gpu.returncode != 0:
        raise RuntimeError(ejecucion_gpu.stderr)
    medicion_gpu = json.loads(ejecucion_gpu.stdout)
    resultados_gpu = medicion_gpu["resultados"]
    origen_gpu = f"Mediciones propias: {medicion_gpu['gpu']}"
    print(origen_gpu)
    print("JAX:", medicion_gpu["jax"], "NumPy:", medicion_gpu["numpy"])
else:
    print("Benchmark omitido; pueden usar la referencia de la celda siguiente.")

#### Referencia docente y gráfico

**Sin GPU compatible:** descomenten las líneas de la celda siguiente y ejecútenla antes del gráfico. **Con mediciones propias:** manténganla comentada para conservar sus resultados.

La referencia se midió el 9 de noviembre de 2026 en el computador del docente: Ryzen 5 220 (6 núcleos y 12 hilos), Radeon 740M integrada, Ubuntu 26.04.1, Python 3.13.13, JAX 0.11.0, ROCm 10.0.0 y NumPy 2.5.3. Se usó el protocolo del benchmark anterior. El escritorio y otros procesos estaban activos; estos tiempos describen esa ejecución y pueden variar. Las entradas son sintéticas, no canciones del catálogo.

El gráfico usa escalas logarítmicas en ambos ejes: separaciones horizontales iguales representan la misma razón entre cantidades de filas, y separaciones verticales iguales, la misma razón entre tiempos. Así podemos mostrar desde una fila hasta un millón. Comparen las dos curvas GPU para observar el costo de entrada y salida. Al comparar NumPy con JAX también cambia la implementación; JAX CPU ayuda a distinguir ese efecto del cambio de dispositivo.

In [ ]:
# Descomenten este bloque únicamente para usar la referencia docente.
# origen_gpu = "Referencia docente: Ryzen 5 220 / Radeon 740M"
# resultados_gpu = [
#     {
#         'filas': 1,
#         'NumPy CPU': 0.005831,
#         'JAX CPU': 0.003787,
#         'GPU residente': 0.05805,
#         'GPU completa': 0.403035,
#     },
#     {
#         'filas': 1000,
#         'NumPy CPU': 0.025408,
#         'JAX CPU': 0.02604,
#         'GPU residente': 0.044064,
#         'GPU completa': 0.334455,
#     },
#     {
#         'filas': 40000,
#         'NumPy CPU': 0.821238,
#         'JAX CPU': 0.292344,
#         'GPU residente': 0.061818,
#         'GPU completa': 0.44807,
#     },
#     {
#         'filas': 200000,
#         'NumPy CPU': 6.376098,
#         'JAX CPU': 1.084107,
#         'GPU residente': 0.158681,
#         'GPU completa': 1.188646,
#     },
#     {
#         'filas': 1000000,
#         'NumPy CPU': 50.376785,
#         'JAX CPU': 4.681953,
#         'GPU residente': 0.603325,
#         'GPU completa': 5.49176,
#     },
# ]

In [ ]:
# Gráfico dado: usa sus mediciones o la referencia descomentada.
import pandas as pd
import plotly.express as px

if resultados_gpu is None:
    print("Midan su GPU o descomenten la referencia para generar el gráfico.")
else:
    tabla_gpu = pd.DataFrame(resultados_gpu).melt(
        id_vars="filas", var_name="Implementación", value_name="Tiempo (ms)"
    )
    figura_gpu = px.line(
        tabla_gpu, x="filas", y="Tiempo (ms)", color="Implementación",
        markers=True, log_x=True, log_y=True,
        labels={"filas": "Número de filas (10 características)"},
        title=origen_gpu,
    )
    figura_gpu.show()

**Datos hipotéticos para el análisis; no son mediciones del laboratorio.** Los tiempos están en milisegundos, excluyen el arranque y el calentamiento, y corresponden a la misma predicción lineal y precisión. La salida se necesita en CPU al terminar.

| Lote | CPU: cálculo | CPU → GPU | GPU: cálculo | GPU → CPU |
|---|---:|---:|---:|---:|
| Pequeño | 20 | 12 | 2 | 9 |
| Grande | 200 | 40 | 10 | 20 |

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (0,5 puntos)</strong>

1. Según sus mediciones o la referencia docente, ¿desde qué tamaño ensayado GPU residente supera a JAX CPU? ¿Se mantiene esa ventaja al incluir entrada y salida? Justifiquen qué alternativa recomendarían a Hernán, citando los tiempos del gráfico y aclarando si usan mediciones propias o la referencia.
2. Para el lote grande, supongan diez predicciones sucesivas sobre los mismos datos: una transferencia de entrada, diez cómputos GPU y una transferencia de salida. Solo se necesita el último resultado en CPU. ¿Cuánto tarda cada alternativa? ¿Qué supuesto dejaría de cumplirse si cada predicción usara datos nuevos?
3. Una matriz tiene 100 millones de filas y diez características `float32` de cuatro bytes. Calculen su tamaño en GB decimales y en GiB (`2**30` bytes). ¿Basta con que una GPU tenga exactamente esa memoria libre? Consideren resultados e intermediarios.
4. Un colega mide solo cuánto tarda en retornar una llamada GPU y anuncia una aceleración enorme. ¿Por qué ese número no permite compararla con el tiempo CPU de la tabla? Expliquen qué incluirían en la medición y cuándo tendría sentido informar también el costo de la primera ejecución para el uso de la radio.
5. Para que quepan más canciones, Toño propone cambiar `float64` por `float32`. ¿Qué compararían con la referencia CPU antes de aceptar el cambio y por qué no exigirían igualdad exacta de todos los decimales?
</div>

`Escriban sus respuestas aquí:`

## 3. Computación Distribuida

### 3.1 Simulación Local (0,5 puntos)

<!-- LORE -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 La cadena de radios de la costa</strong>

El catálogo de Faro Papaya ya no es el único que interesa a Hernán: otras radios comunitarias de la costa quieren comparar su programación y conocer la valencia promedio por género de toda la cadena. Cada radio conservará sus archivos. Reunirlos en el computador de Hernán podría superar su memoria, incluso leyendo solo género y valencia; además, para obtener ese promedio no necesitan trasladar cada canción.

Después de revisar los costos de mover datos, Hernán propone que cada radio haga sus cuentas y envíe un resumen. «Ya nos costó abrir nuestro catálogo; no traigamos todos los demás de una vez», le dice a Toño. Ustedes ensayarán cómo combinar esos resúmenes con los archivos disponibles y comprobarán que el resultado represente al catálogo completo, aunque los trabajadores reciban cantidades distintas de archivos.
</div>

<!-- DEFINICIÓN -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Particiones y reducción</strong>

En **computación distribuida**, el trabajo se reparte entre procesos que pueden ejecutarse en máquinas diferentes y comunicarse mediante mensajes. Cada máquina que participa es un **nodo**. Un **trabajador** (*worker*) es el proceso o componente que ejecuta las tareas asignadas, y un nodo puede alojar varios trabajadores. El **coordinador** organiza el reparto y reúne los resultados. Un **clúster** es un conjunto de nodos conectados que trabajan juntos: tres trabajadores en un computador no equivalen a tres máquinas.

Una **partición** contiene una parte de los datos. Si cada trabajador puede resumir su partición, el coordinador recibe esos resúmenes en lugar de todos los registros. Llamamos **reducción** a la operación que los combina para producir el resultado global. Lo que debe contener cada resumen depende de la pregunta que queremos responder.

Pensemos en varias bibliotecas que necesitan conocer la cantidad promedio de páginas de sus libros sin enviar todos sus registros. Cada sede puede informar la suma de páginas y la cantidad de libros. Antes de programar, anticipen si recibir solo el promedio de cada sede permitiría recuperar el promedio global cuando las cantidades de libros son distintas.

Aquí simularemos tres trabajadores **en un único proceso y de forma secuencial**. El ensayo permite estudiar cómo combinar sus resultados; no ejecuta un clúster ni mide comunicación por red. En un despliegue real, cada trabajador tendría que acceder a sus archivos desde su máquina: una ruta local del coordinador no se vuelve accesible automáticamente para las demás.
<img src="assets/nodos-trabajadores-coordinador.svg" alt="Dos nodos contienen tres trabajadores que envían resúmenes al coordinador. El notebook simula esos trabajadores secuencialmente dentro de un solo proceso." style="display:block; margin:12px auto; width:960px; max-width:100%; height:auto;">

</div>

**Código dado:** reparte los 20 archivos entre tres trabajadores simulados con 1, 4 y 15 archivos. Lee solo `genre` y `valence` y entrega un resumen por género con `suma` y `cantidad`. Cada fila de entrada es una canción y cada fila del resumen corresponde a un género en una partición. El ensayo comprueba que no haya nulos en las columnas utilizadas y que cada archivo se asigne una sola vez.

In [ ]:
paths = sorted(RAW_DIR.glob("*.parquet"))
assert len(paths) == 20
particiones = [paths[:1], paths[1:5], paths[5:]]
asignados = []
resumenes = []
for archivos in particiones:
    tablas = []
    for path in archivos:
        tablas.append(pd.read_parquet(path, columns=["genre", "valence"]))
        asignados.append(path)
    datos_locales = pd.concat(tablas, ignore_index=True)
    assert not datos_locales.isna().any().any()
    resumen = datos_locales.groupby("genre")["valence"].agg(
        suma="sum", cantidad="count"
    )
    resumenes.append(resumen)

assert sorted(asignados) == paths
for trabajador, resumen in enumerate(resumenes, start=1):
    print(
        f"Trabajador {trabajador}: {resumen['cantidad'].sum()} canciones; "
        f"envía {len(resumen)} filas de resumen."
    )

<!-- EJERCICIO -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #06b6d4; background:rgba(6,182,212,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💻 Ejercicio 3.1: combinar los resúmenes (0,2 puntos)</strong>

Cada elemento de `resumenes` contiene las cuentas de un trabajador: una fila por género, con `suma` y `cantidad`. Falta reunir esas cuentas en un resultado para toda la cadena. Trabajen con los resúmenes disponibles; esta etapa no requiere recuperar los registros individuales.

- [ ] Construir `resumen_global` en la celda siguiente, con una fila por género, índice `genre` y columnas `suma`, `cantidad` y `promedio`.
- [ ] Conservar la cantidad total de canciones y calcular el promedio a partir de las sumas y cantidades, sin volver a leer los archivos ni reunir sus canciones. Pueden usar `pd.concat` y `groupby`.
- [ ] Ejecutar la comprobación provista: deben conservarse las 200.000 canciones y los promedios por género deben coincidir con el cálculo centralizado, con tolerancia absoluta `1e-12` y relativa `0`.
</div>

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan resumen_global antes de continuar.")

**Código dado — comprobación centralizada:** la referencia usa `df_train`, el catálogo completo cargado en 2.1. La comparación ordena los géneros para alinear sus filas.

In [ ]:
import numpy as np

referencia_generos = df_train.groupby("genre")["valence"].mean().sort_index()
assert resumen_global.index.is_unique
assert resumen_global["cantidad"].sum() == len(df_train) == 200_000
assert resumen_global.index.sort_values().equals(referencia_generos.index)
np.testing.assert_allclose(
    resumen_global["promedio"].sort_index(),
    referencia_generos,
    atol=1e-12,
    rtol=0,
)
print("200.000 canciones conservadas; promedios por género equivalentes.")

<!-- PREGUNTA -->
<div style="box-sizing:border-box; max-width:100%; overflow-wrap:break-word; padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas (0,3 puntos)</strong>

1. Para un mismo género, una partición tiene una canción con valencia 0,9 y otra tiene nueve canciones con valencia promedio 0,1. Calculen el promedio de promedios y el promedio global correcto. ¿Qué información de `resumenes` evita el error?
2. Si cada trabajador estuviera en otra máquina, ¿qué archivos necesitaría poder leer y qué datos enviaría al coordinador? Comparen enviar los resúmenes con enviar todas las filas de género y valencia. ¿Por qué `ProcessPoolExecutor` local no demuestra que el trabajo ya se ejecutó en varias máquinas?
3. El coordinador recibe dos veces el resumen de una partición después de un reintento. ¿Qué falla concreta produciría sumarlo dos veces y cómo lo evitarían? ¿Qué consecuencia tendría que un trabajador recibiera muchos más datos que los demás?
</div>

`Escriban sus respuestas aquí:`

---

Cualquier duda sobre el enunciado o la corrección va al foro de U-Cursos. La entrega cierra en la fecha publicada en el calendario del curso.